# Task 2: remaining code — CPU preparation

Keep Hardware accelerator **None**. Run the code cell and upload **task2-chunk3.zip**. Wait for **GPU JOBS READY** and the Drive flush confirmation.

In [ ]:
from pathlib import Path
from uuid import uuid4
import hashlib
import json
import shutil
import signal
import subprocess
import sys
import tempfile
import time
import zipfile


def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(2**20), b''):
            h.update(block)
    return h.hexdigest()


def safe_extract(z, destination):
    destination = Path(destination).resolve()
    names = set()
    for item in z.infolist():
        target = (destination / item.filename).resolve()
        if destination not in target.parents or item.filename in names or (item.external_attr >> 16) & 0o170000 == 0o120000:
            raise RuntimeError('Unsafe or duplicate archive member')
        names.add(item.filename)
    z.extractall(destination)


def run_visible(command, cwd=None):
    process = subprocess.Popen(list(map(str, command)), cwd=cwd, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
    except KeyboardInterrupt:
        # The GPU orchestrator stops its child process and saves completed work.
        process.send_signal(signal.SIGINT)
        remaining, _ = process.communicate()
        print(remaining, end='', flush=True)
        code = process.returncode
    except BaseException:
        process.terminate()
        process.wait()
        raise
    if code:
        raise RuntimeError('Process failed; keep the runtime available and inspect the printed error.')


def latest_receipt(pattern, status, override=''):
    choices = [Path(override)] if override else sorted(Path('/content/drive/MyDrive/ATML-Assignment-2/task2').glob(pattern), reverse=True)
    for path in choices:
        record = json.loads(path.read_text())
        if record['status'] == status:
            return path, record
    raise RuntimeError('No completed preparation receipt found. Run the CPU preparation notebook first.')


def restore_project(ready, repo):
    spec = ready['project_archive']
    repo = Path(repo)
    with tempfile.TemporaryDirectory(prefix='task2_bootstrap_') as tmp:
        local = Path(tmp) / 'project.zip'
        shutil.copyfile(spec['path'], local)
        if file_sha(local) != spec['sha256']:
            raise RuntimeError('Saved project archive checksum differs')
        with zipfile.ZipFile(local) as z:
            manifest = json.loads(z.read('TASK2_BACKUP_MANIFEST.json'))['files_sha256']
            expected = set(manifest) | {'TASK2_BACKUP_MANIFEST.json'}
            if set(z.namelist()) != expected or len(z.namelist()) != len(expected):
                raise RuntimeError('Project archive inventory differs')
            for name, digest in manifest.items():
                if hashlib.sha256(z.read(name)).hexdigest() != digest:
                    raise RuntimeError('Project member checksum differs: ' + name)
            extracted = Path(tmp) / 'extracted'
            safe_extract(z, extracted)
        for name, digest in manifest.items():
            target = repo / name
            if target.exists() and file_sha(target) != digest:
                raise RuntimeError('Existing project file differs; use a fresh runtime: ' + name)
        for name in manifest:
            target = repo / name
            if not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(extracted / name, target)

# Hardware accelerator: None. Optional exact Chunk 2 receipt path:
PREPARATION_RECEIPT = ''
import torch
from google.colab import drive, files
if torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > Hardware accelerator: None.')
drive.mount('/content/drive')
receipt, preparation = latest_receipt('cpu_prepared/*/preparation.json', 'CPU_PREPARATION_VERIFIED', PREPARATION_RECEIPT)
print('Using verified Chunk 2 receipt:', receipt, flush=True)
print('Upload task2-chunk3.zip (the small remaining-code package).', flush=True)
uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError('Upload exactly one ZIP package.')
filename = next(iter(uploaded))
package = Path('/content') / ('task2_chunk3_' + uuid4().hex[:8])
package.mkdir()
with zipfile.ZipFile(filename) as z:
    safe_extract(z, package)
if not (package / 'setup_chunk3.py').is_file():
    raise RuntimeError('Expected the remaining-code ZIP with setup_chunk3.py at its root.')
manifest = json.loads((package / 'PACKAGE_SHA256.json').read_text())
for name, digest in manifest.items():
    if file_sha(package / name) != digest:
        raise RuntimeError('Uploaded package file differs: ' + name)
run_visible([sys.executable, '-u', package / 'setup_chunk3.py', '--preparation', receipt])
drive.flush_and_unmount()
print('GPU JOBS READY. Drive flushed. Next: open Task2_GPU_Run.ipynb and select A100.', flush=True)
